# GARCH Baseline

GARCH(1,1) with Student-t innovations + constant mean, plus a calibration quantile over `|y - mu| / sigma_t`. This is a conformal-flavoured GARCH: the interval is `[mu +/- q_hat * sigma_next]` where `q_hat = quantile(1 - alpha)` over calibration standardized residuals. Serves as the secondary industry baseline named in the proposal.

Saves CSVs under `results/garch_native_<ticker>_<regime>.csv` with `model="garch"` and `wrapper="native"`.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in (_here, *_here.parents):
    if (_p / "requirements.txt").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

In [ ]:
import numpy as np
import pandas as pd

from src.analysis._shared import (
    ensure_data,
    iterate_experiments,
    walk_forward,
    save_intervals,
)
from src.conformal.garch import Garch
from src.data.data import Ticker

In [ ]:
CONFIDENCE_LEVEL = 0.90  # alpha = 0.10, matches Metrics.md
MODEL_NAME = "garch"
WRAPPER_NAME = "native"

np.random.seed(0)

## Load data

In [ ]:
data = ensure_data()
print(f"{len(data):,} rows  |  {data.index.min().date()} -> {data.index.max().date()}")
data.head()

## Smoke test: one split

In [ ]:
smoke_split = next(
    s for s in iterate_experiments()
    if s.regime == "2020" and s.ticker == Ticker.McDonalds
)
train_df = smoke_split.slice_train(data)
calib_df = smoke_split.slice_calib(data)
full_df  = smoke_split.slice_full(data)

wrapper = Garch(
    label_col_name=smoke_split.label_col,
    num_lags=1,
    confidence_level=CONFIDENCE_LEVEL,
)
wrapper.fit_and_calibrate(train_df, calib_df)

intervals = walk_forward(
    wrapper, full_df,
    test_start=smoke_split.test_start,
    test_end=smoke_split.test_end,
)
save_intervals(intervals, MODEL_NAME, WRAPPER_NAME, smoke_split.ticker, smoke_split.regime)

cov = ((intervals["r"] >= intervals["l"]) & (intervals["r"] <= intervals["u"])).mean()
print(f"GARCH  cov={cov:.3f}  mean_w={(intervals['u']-intervals['l']).mean():.4f}  n={len(intervals)}")

## Full grid

In [ ]:
RUN_FULL_GRID = True

if RUN_FULL_GRID:
    results = []
    for split in iterate_experiments():
        train_df = split.slice_train(data)
        calib_df = split.slice_calib(data)
        full_df  = split.slice_full(data)

        wrapper = Garch(
            label_col_name=split.label_col,
            num_lags=1,
            confidence_level=CONFIDENCE_LEVEL,
        )
        wrapper.fit_and_calibrate(train_df, calib_df)
        intervals = walk_forward(
            wrapper, full_df,
            test_start=split.test_start,
            test_end=split.test_end,
        )
        save_intervals(intervals, MODEL_NAME, WRAPPER_NAME, split.ticker, split.regime)

        covered = (intervals["r"] >= intervals["l"]) & (intervals["r"] <= intervals["u"])
        widths  = intervals["u"] - intervals["l"]
        results.append({
            "regime":  split.regime,
            "ticker":  split.ticker.value,
            "coverage": covered.mean(),
            "mean_width": widths.mean(),
            "inf_days": int(np.isinf(widths).sum()),
        })
        print(f"  {split.regime} {split.ticker.value:5s}: cov={covered.mean():.3f}  mean_w={widths.mean():.4f}")

    pd.DataFrame(results)